In [1]:
import pandas as pd
import os
# from transformers import BertTokenizer, BertModel
from bert_score import BERTScorer
import re
import pickle
import nltk
from nltk.translate import meteor
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
import math

/Users/isabel/anaconda3/envs/entityRecognitionNotes/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /Users/isabel/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nlt

True

In [3]:
gpt3_load = './generatedDataFromGoogleDrive_gpt3/data/'
gpt4_load = './generatedDataFromGoogleDrive_gpt4/data/'
gpt3_save_bert = './metrics/gpt3_F1_BERT_scores.pkl'
gpt4_save_bert = './metrics/gpt4_F1_BERT_scores.pkl'
gpt3_save_meteor = './metrics/gpt3_meteor_scores.pkl'
gpt4_save_meteor = './metrics/gpt4_meteor_scores.pkl'
gpt3_save_sentiment = './metrics/gpt3_sentiment_scores.pkl'
gpt4_save_sentiment = './metrics/gpt4_sentiment_scores.pkl'

# Contextual Similarity - BERTScore

In [4]:
def bertScore(loadingFile, savingFile):
    # load all files from data folder (generated in Google Colab)
    met_dfs = {}
    unmet_dfs = {}
    for file in os.listdir(loadingFile):
        if '.csv' in file:
            
            if 'unmet' in file:
                unmet_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
            else:
                met_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
    nurse_notes = pd.read_excel('../../fake_notes.xlsx')

    score_dictionary = {}
    scorer = BERTScorer(model_type="bert-base-uncased")
    for i in range(len(nurse_notes)):
        word = ""
        # BERTScore penalizes punctuation, we should take this into account - https://aclanthology.org/2023.findings-acl.381.pdf
        if i < 5:
            candidate =  [re.sub(r'[^\w\s/]', '', i) for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            word = "Met"
        else:
            candidate =  [re.sub(r'[^\w\s/]', '', i) for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            word = "Unmet"
        reference = [re.sub(r'[^\w\s/]', '', nurse_notes['Note'][i])] * (len(candidate))
        try:
            P, R, F1 = scorer.score(candidate, reference)
        except Exception as e:
            print(e)
        # save the F1 values, as these are recommended for use by the BERTScore authors - http://arxiv.org/abs/1904.09675
        score_dictionary[f'{i}_{word.lower()}'] = F1.mean()
        print(f"{word} {i} F1 Score: {F1.mean()}")

    # put scores in a pickle file
    with open(savingFile, 'wb') as f:
        pickle.dump(score_dictionary, f)
    

In [5]:
bertScore(gpt4_load, gpt4_save_bert)

Met 0 F1 Score: 0.5635975003242493
Met 1 F1 Score: 0.5388426184654236
Met 2 F1 Score: 0.5419347882270813
Met 3 F1 Score: 0.6021817922592163
Met 4 F1 Score: 0.5532594323158264
Unmet 5 F1 Score: 0.4842208921909332
Unmet 6 F1 Score: 0.5917355418205261
Unmet 7 F1 Score: 0.5667903423309326
Unmet 8 F1 Score: 0.5572194457054138
Unmet 9 F1 Score: 0.5282754898071289


In [6]:
# load scores from pickle file - gpt4
with open(gpt4_save_bert, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)

loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()
print(f"Average BERTScore Met Notes GPT4: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore Unmet Notes GPT4: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore All Notes GPT4: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")



{'0_met': tensor(0.5636), '1_met': tensor(0.5388), '2_met': tensor(0.5419), '3_met': tensor(0.6022), '4_met': tensor(0.5533), '5_unmet': tensor(0.4842), '6_unmet': tensor(0.5917), '7_unmet': tensor(0.5668), '8_unmet': tensor(0.5572), '9_unmet': tensor(0.5283)}
Average BERTScore Met Notes GPT4: 0.5599632263183594
Average BERTScore Unmet Notes GPT4: 0.5456483364105225
Average BERTScore All Notes GPT4: 0.5528057813644409


In [7]:
bertScore(gpt3_load, gpt3_save_bert)

Met 0 F1 Score: 0.5719854235649109
Met 1 F1 Score: 0.4812847077846527
Met 2 F1 Score: 0.5140470266342163
Met 3 F1 Score: 0.5805385112762451
Met 4 F1 Score: 0.6390343904495239
Unmet 5 F1 Score: 0.5105171799659729
Unmet 6 F1 Score: 0.5724819302558899
Unmet 7 F1 Score: 0.5182159543037415
Unmet 8 F1 Score: 0.4988231658935547
Unmet 9 F1 Score: 0.4897165596485138


In [8]:
# load scores from pickle file - gpt3
with open(gpt3_save_bert, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)
loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()

print(f"Average BERTScore Met Notes GPT3: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore Unmet Notes GPT3: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore All Notes GPT3: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")


{'0_met': tensor(0.5720), '1_met': tensor(0.4813), '2_met': tensor(0.5140), '3_met': tensor(0.5805), '4_met': tensor(0.6390), '5_unmet': tensor(0.5105), '6_unmet': tensor(0.5725), '7_unmet': tensor(0.5182), '8_unmet': tensor(0.4988), '9_unmet': tensor(0.4897)}
Average BERTScore Met Notes GPT3: 0.5573779940605164
Average BERTScore Unmet Notes GPT3: 0.5179509520530701
Average BERTScore All Notes GPT3: 0.5376644730567932


# Lexical Overlap Metric - METEOR

In [9]:
def calculate_meteor(loadingFile, savingFile):
    # load all files from data folder (generated in Google Colab)
    met_dfs = {}
    unmet_dfs = {}
    for file in os.listdir(loadingFile):
        if '.csv' in file:
            
            if 'unmet' in file:
                unmet_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
            else:
                met_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
    nurse_notes = pd.read_excel('../../fake_notes.xlsx')

    score_dictionary = {}
    for i in range(len(nurse_notes)):
        meteor_scores = []
        word = ""
        if i < 5:
            candidates =  [re.sub(r'[^\w\s/]', '', i) for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            word = "Met"
        else:
            candidates =  [re.sub(r'[^\w\s/]', '', i) for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            word = "Unmet"
        reference = re.sub(r'[^\w\s/]', '', nurse_notes['Note'][i])
        for candidate in candidates:
            output_score = meteor([word_tokenize(candidate)], word_tokenize(reference))
            meteor_scores.append(output_score)
        try:
            score_dictionary[f'{i % (len(nurse_notes) // 2)}_{word}'] = (sum(meteor_scores)) / (len(meteor_scores))
        except:
            score_dictionary[f'{i % (len(nurse_notes) // 2)}_{word}'] = 0
    print(score_dictionary)

    # put scores in a pickle file
    with open(savingFile, 'wb') as f:
        pickle.dump(score_dictionary, f)

In [10]:
calculate_meteor(gpt3_load, gpt3_save_meteor)

{'0_Met': 0.21772189251089138, '1_Met': 0.1132496931667288, '2_Met': 0.10711144811973017, '3_Met': 0.20546261648477904, '4_Met': 0.3296554223858355, '0_Unmet': 0.0758222336111236, '1_Unmet': 0.19359328194327116, '2_Unmet': 0.11082067703851463, '3_Unmet': 0.12395500980458082, '4_Unmet': 0.14510185184502097}


In [11]:
# load scores from pickle file - gpt3
with open(gpt3_save_meteor, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)
loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()

print(f"Average METEOR Score Met Notes GPT3: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score Unmet Notes GPT3: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score All Notes GPT3: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")

{'0_Met': 0.21772189251089138, '1_Met': 0.1132496931667288, '2_Met': 0.10711144811973017, '3_Met': 0.20546261648477904, '4_Met': 0.3296554223858355, '0_Unmet': 0.0758222336111236, '1_Unmet': 0.19359328194327116, '2_Unmet': 0.11082067703851463, '3_Unmet': 0.12395500980458082, '4_Unmet': 0.14510185184502097}
Average METEOR Score Met Notes GPT3: 0.19464021453359298
Average METEOR Score Unmet Notes GPT3: 0.12985861084850223
Average METEOR Score All Notes GPT3: 0.1622494126910476


In [12]:
calculate_meteor(gpt4_load, gpt4_save_meteor)

{'0_Met': 0.16846800083637578, '1_Met': 0.13106798250069163, '2_Met': 0.15105134367800377, '3_Met': 0.19962566489649747, '4_Met': 0.15689171434220603, '0_Unmet': 0.05908176207819897, '1_Unmet': 0.17000218067344794, '2_Unmet': 0.14443434661724852, '3_Unmet': 0.1709802927366372, '4_Unmet': 0.1829054586832099}


In [13]:
# load scores from pickle file - gpt4
with open(gpt4_save_meteor, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)
loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()

print(f"Average METEOR Score Met Notes GPT4: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score Unmet Notes GPT4: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score All Notes GPT4: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")

{'0_Met': 0.16846800083637578, '1_Met': 0.13106798250069163, '2_Met': 0.15105134367800377, '3_Met': 0.19962566489649747, '4_Met': 0.15689171434220603, '0_Unmet': 0.05908176207819897, '1_Unmet': 0.17000218067344794, '2_Unmet': 0.14443434661724852, '3_Unmet': 0.1709802927366372, '4_Unmet': 0.1829054586832099}
Average METEOR Score Met Notes GPT4: 0.16142094125075496
Average METEOR Score Unmet Notes GPT4: 0.1454808081577485
Average METEOR Score All Notes GPT4: 0.15345087470425173


# Sentiment Analysis - TextBlob

In [14]:
def preprocess_text(text):
    text = re.sub(r'[^\w\s/]', '', text)
    # tokenize
    tokens = word_tokenize(text.lower())
    # remove stop words
    filtered_tokens = [token for token in tokens if token not in stopwords.words('english')]
    # lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    # join the tokens back into a string
    processed_text = ' '.join(lemmatized_tokens)
    return processed_text

In [15]:
def sentiment_interpreter(sentiment):
    sentiment = round(sentiment, 2)
    if sentiment > 0.5:
        return "positive"
    elif sentiment < -0.5: 
        return "negative"
    else:
        return "neutral"

In [16]:
def calculate_sentiment(loadingFile, savingFile):
    # load all files from data folder (generated in Google Colab)
    met_dfs = {}
    unmet_dfs = {}
    for file in os.listdir(loadingFile):
        if '.csv' in file:
            
            if 'unmet' in file:
                unmet_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
            else:
                met_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
    nurse_notes = pd.read_excel('../../fake_notes.xlsx')

    score_dictionary = {}
    for i in range(len(nurse_notes)):
        nurse_note = preprocess_text(nurse_notes['Note'][i])
        nurse_blob = TextBlob(nurse_note)
        nurse_sentiment = nurse_blob.sentences[0].sentiment.polarity
        nurse_subjectivity = nurse_blob.sentences[0].sentiment.subjectivity
        word = ""
        if i < 5:
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.polarity for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            try:
                generated_sentiment = (sum(candidates))/ len(candidates)
            except:
                generated_sentiment = 0
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.subjectivity for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            try:
                generated_subjectivity = (sum(candidates))/ len(candidates)
            except:
                generated_subjectivity = 0
            word = "Met"
        else:
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.polarity for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            try:
                generated_sentiment = (sum(candidates))/ len(candidates)
            except:
                generated_sentiment = 0
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.subjectivity for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            try:
                generated_subjectivity = (sum(candidates))/ len(candidates)
            except:
                generated_subjectivity = 0
            word = "Unmet"
        score_dictionary[f'{i % (len(nurse_notes) // 2)}_{word}'] = {"Nurse Sentiment": nurse_sentiment, "Generated Sentiment": generated_sentiment, "Absolute Difference in Sentiment": math.sqrt((nurse_sentiment-generated_sentiment) ** 2), "Nurse Subjectivity": nurse_subjectivity , "Generated Subjectivity": generated_subjectivity, "Absolute Difference in Subjectivity":math.sqrt((nurse_subjectivity-generated_subjectivity) ** 2)}
    print(score_dictionary)
    # put scores in a pickle file
    with open(savingFile, 'wb') as f:
        pickle.dump(score_dictionary, f)

In [17]:
calculate_sentiment(gpt3_load, gpt3_save_sentiment)

{'0_Met': {'Nurse Sentiment': 0.023333333333333317, 'Generated Sentiment': 0.26634444444444444, 'Absolute Difference in Sentiment': 0.24301111111111112, 'Nurse Subjectivity': 0.4866666666666667, 'Generated Subjectivity': 0.514868253968254, 'Absolute Difference in Subjectivity': 0.028201587301587283}, '1_Met': {'Nurse Sentiment': 0.43333333333333335, 'Generated Sentiment': 0.1185, 'Absolute Difference in Sentiment': 0.31483333333333335, 'Nurse Subjectivity': 0.5416666666666666, 'Generated Subjectivity': 0.3495576923076923, 'Absolute Difference in Subjectivity': 0.19210897435897434}, '2_Met': {'Nurse Sentiment': 0.24444444444444444, 'Generated Sentiment': 0.20708531746031747, 'Absolute Difference in Sentiment': 0.03735912698412697, 'Nurse Subjectivity': 0.2638888888888889, 'Generated Subjectivity': 0.43933333333333335, 'Absolute Difference in Subjectivity': 0.17544444444444446}, '3_Met': {'Nurse Sentiment': 0.08484848484848484, 'Generated Sentiment': 0.22226785714285713, 'Absolute Differ

In [18]:
def sentiment_parser(loaded_scores):
    nurse_met_sentiment = 0
    generated_met_sentiment = 0
    nurse_met_subjectivity = 0
    generated_met_subjectivity = 0
    met_count = 0
    nurse_unmet_sentiment = 0
    generated_unmet_sentiment = 0
    nurse_unmet_subjectivity = 0
    generated_unmet_subjectivity = 0
    unmet_count = 0
    for score in loaded_scores:
        if 'Unmet' in score:
            nurse_unmet_sentiment += loaded_scores[score]['Nurse Sentiment']
            generated_unmet_sentiment += loaded_scores[score]['Generated Sentiment']
            nurse_unmet_subjectivity += loaded_scores[score]['Nurse Subjectivity']
            generated_unmet_subjectivity += loaded_scores[score]['Generated Subjectivity']
            unmet_count += 1

        elif 'Met' in score:
            # print(loaded_scores[score]['Generated Sentiment'])
            nurse_met_sentiment += loaded_scores[score]['Nurse Sentiment']
            generated_met_sentiment += loaded_scores[score]['Generated Sentiment']
            nurse_met_subjectivity += loaded_scores[score]['Nurse Subjectivity']
            generated_met_subjectivity += loaded_scores[score]['Generated Subjectivity']
            met_count += 1
    # met note scores
    print("\n------------\nMet Notes\n------------\n")
    print(f"Nurse Met Sentiment: {nurse_met_sentiment / met_count}")
    print(f"Generated Met Sentiment: {generated_met_sentiment / met_count}")
    print(f"Difference in Met Sentiment: {math.sqrt(((nurse_met_sentiment / met_count)-(generated_met_sentiment / met_count)) ** 2)}")
    print(f"Nurse Met Subjectivity: {nurse_met_subjectivity / met_count}")
    print(f"Generated Met Subjectivity: {generated_met_subjectivity / met_count}")
    print(f"Difference in Met Subjectivity: {math.sqrt(((nurse_met_subjectivity / met_count)-(generated_met_subjectivity / met_count)) ** 2)}")

    # unmet note scores
    print("\n------------\nUnmet Notes\n------------\n")
    print(f"Nurse Unmet Sentiment: {nurse_unmet_sentiment / unmet_count}")
    print(f"Generated Unmet Sentiment: {generated_unmet_sentiment / unmet_count}")
    print(f"Difference in Unmet Sentiment: {math.sqrt(((nurse_unmet_sentiment / unmet_count)-(generated_unmet_sentiment / unmet_count)) ** 2)}")
    print(f"Nurse Unmet Subjectivity: {nurse_unmet_subjectivity / unmet_count}")
    print(f"Generated Unmet Subjectivity: {generated_unmet_subjectivity / unmet_count}")
    print(f"Difference in Unmet Subjectivity: {math.sqrt(((nurse_unmet_subjectivity / unmet_count)-(generated_unmet_subjectivity / unmet_count)) ** 2)}")


In [19]:
# load scores from pickle file - gpt4
with open(gpt3_save_sentiment, 'rb') as f:
    loaded_scores = pickle.load(f)
print("GPT3")
print(loaded_scores)
sentiment_parser(loaded_scores=loaded_scores)

GPT3
{'0_Met': {'Nurse Sentiment': 0.023333333333333317, 'Generated Sentiment': 0.26634444444444444, 'Absolute Difference in Sentiment': 0.24301111111111112, 'Nurse Subjectivity': 0.4866666666666667, 'Generated Subjectivity': 0.514868253968254, 'Absolute Difference in Subjectivity': 0.028201587301587283}, '1_Met': {'Nurse Sentiment': 0.43333333333333335, 'Generated Sentiment': 0.1185, 'Absolute Difference in Sentiment': 0.31483333333333335, 'Nurse Subjectivity': 0.5416666666666666, 'Generated Subjectivity': 0.3495576923076923, 'Absolute Difference in Subjectivity': 0.19210897435897434}, '2_Met': {'Nurse Sentiment': 0.24444444444444444, 'Generated Sentiment': 0.20708531746031747, 'Absolute Difference in Sentiment': 0.03735912698412697, 'Nurse Subjectivity': 0.2638888888888889, 'Generated Subjectivity': 0.43933333333333335, 'Absolute Difference in Subjectivity': 0.17544444444444446}, '3_Met': {'Nurse Sentiment': 0.08484848484848484, 'Generated Sentiment': 0.22226785714285713, 'Absolute D

In [20]:
calculate_sentiment(gpt4_load, gpt4_save_sentiment)

{'0_Met': {'Nurse Sentiment': 0.023333333333333317, 'Generated Sentiment': 0.041125834235209234, 'Absolute Difference in Sentiment': 0.017792500901875917, 'Nurse Subjectivity': 0.4866666666666667, 'Generated Subjectivity': 0.36602840909090906, 'Absolute Difference in Subjectivity': 0.12063825757575763}, '1_Met': {'Nurse Sentiment': 0.43333333333333335, 'Generated Sentiment': 0.1322940355940356, 'Absolute Difference in Sentiment': 0.3010392977392977, 'Nurse Subjectivity': 0.5416666666666666, 'Generated Subjectivity': 0.4473130314130314, 'Absolute Difference in Subjectivity': 0.09435363525363522}, '2_Met': {'Nurse Sentiment': 0.24444444444444444, 'Generated Sentiment': 0.10764541847041846, 'Absolute Difference in Sentiment': 0.136799025974026, 'Nurse Subjectivity': 0.2638888888888889, 'Generated Subjectivity': 0.38868076183076183, 'Absolute Difference in Subjectivity': 0.12479187294187294}, '3_Met': {'Nurse Sentiment': 0.08484848484848484, 'Generated Sentiment': 0.09825168268239697, 'Abs

In [21]:
# load scores from pickle file - gpt4
with open(gpt4_save_sentiment, 'rb') as f:
    loaded_scores = pickle.load(f)
print("GPT4")
print(loaded_scores)
sentiment_parser(loaded_scores=loaded_scores)

GPT4
{'0_Met': {'Nurse Sentiment': 0.023333333333333317, 'Generated Sentiment': 0.041125834235209234, 'Absolute Difference in Sentiment': 0.017792500901875917, 'Nurse Subjectivity': 0.4866666666666667, 'Generated Subjectivity': 0.36602840909090906, 'Absolute Difference in Subjectivity': 0.12063825757575763}, '1_Met': {'Nurse Sentiment': 0.43333333333333335, 'Generated Sentiment': 0.1322940355940356, 'Absolute Difference in Sentiment': 0.3010392977392977, 'Nurse Subjectivity': 0.5416666666666666, 'Generated Subjectivity': 0.4473130314130314, 'Absolute Difference in Subjectivity': 0.09435363525363522}, '2_Met': {'Nurse Sentiment': 0.24444444444444444, 'Generated Sentiment': 0.10764541847041846, 'Absolute Difference in Sentiment': 0.136799025974026, 'Nurse Subjectivity': 0.2638888888888889, 'Generated Subjectivity': 0.38868076183076183, 'Absolute Difference in Subjectivity': 0.12479187294187294}, '3_Met': {'Nurse Sentiment': 0.08484848484848484, 'Generated Sentiment': 0.09825168268239697,